# Healthcare Outcomes — Analysis Notebook

Interactive exploration of the clinical marts and the governance overhead experiment.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import snowflake.connector

load_dotenv(Path('..').resolve() / '.env')
sns.set_theme(style='whitegrid')

def conn(role=None):
    return snowflake.connector.connect(
        account=os.environ['SNOWFLAKE_ACCOUNT'],
        user=os.environ['SNOWFLAKE_USER'],
        password=os.environ['SNOWFLAKE_PASSWORD'],
        role=role or os.environ.get('SNOWFLAKE_ROLE','DBT_ROLE'),
        warehouse=os.environ.get('SNOWFLAKE_WAREHOUSE','DBT_WH_XS'),
        database=os.environ.get('SNOWFLAKE_DATABASE','HEALTHCARE_DB'),
    )

def q(sql, role=None):
    c = conn(role); cur = c.cursor(); cur.execute(sql)
    cols = [d[0].lower() for d in cur.description]
    df = pd.DataFrame(cur.fetchall(), columns=cols)
    cur.close(); c.close(); return df

## Clinical: readmission risk distribution

In [ ]:
q('SELECT risk_band, COUNT(*) AS encounters, AVG(risk_score) AS avg_risk FROM MARTS.MART_READMISSION_RISK GROUP BY 1 ORDER BY 1')

## Length of stay by department

In [ ]:
q('SELECT department, AVG(avg_los_days) AS los_days, AVG(readmission_rate) AS readm FROM MARTS.MART_LENGTH_OF_STAY_ANALYTICS GROUP BY 1 ORDER BY 2 DESC LIMIT 15')

## Governance — what does each role see?

In [ ]:
# As ANALYST: PHI columns should be redacted
q('SELECT first_name, last_name, mrn, date_of_birth, phone_number FROM MARTS.DIM_PATIENT LIMIT 5', role='ANALYST_ROLE')

In [ ]:
# As CLINICIAN: name + MRN visible, DOB year-only
q('SELECT first_name, last_name, mrn, date_of_birth, phone_number FROM MARTS.DIM_PATIENT LIMIT 5', role='CLINICIAN_ROLE')

## Experiment: governance overhead

In [ ]:
m = q('SELECT * FROM META.experiment_metrics')
m.groupby(['condition','metric_name'])['metric_value'].agg(['mean','std','count']).reset_index()

In [ ]:
wide = m.pivot_table(index=['run_uuid','metric_name'], columns='condition',
                    values='metric_value', aggfunc='first').reset_index()
rows = []
for metric in ['mean_latency_ms','p95_latency_ms','credits_used','loc_redaction']:
    sub = wide[wide['metric_name']==metric].dropna(subset=['BASELINE_VIEWS','INTERVENTION_POLICIES'])
    if len(sub) < 5: continue
    b = sub['BASELINE_VIEWS']; i = sub['INTERVENTION_POLICIES']
    diff = (i - b).values
    if diff.std(ddof=1) == 0:
        rows.append({'metric': metric, 'mean_diff': diff.mean(), 'p_value': None,
                     'cohens_d': None, 'significant': False, 'note': 'zero variance'})
        continue
    w_stat, p = stats.wilcoxon(i, b)
    rows.append({'metric': metric, 'mean_diff': diff.mean(),
                 'p_value': p, 'cohens_d': diff.mean()/(diff.std(ddof=1)+1e-12),
                 'significant': p < 0.05})
pd.DataFrame(rows)